In [1]:
import os
import numpy as np
import nibabel as nib
from nilearn.image import resample_to_img
import pandas as pd

# --- Paths (update if typo in filenames) ---
base = r"N:\Experimental_Data\yujunchen\projects\IAPS_Searchlight\outputs\tBrainmap\main"
paths = {
    "emotion": os.path.join(base, "tmap_beh60_p05_v2.nii.gz"),
    "scene"  : os.path.join(base, "tmap_clipvit_p001_fisherz_fdr01.nii.gz"),
    "gist"   : os.path.join(base, "tmap_gist_norm_p001.nii.gz"),
}

# --- Helpers ---
def load_bin(path, ref_img=None):
    img = nib.load(path)
    if ref_img is not None:
        img = resample_to_img(img, ref_img, interpolation="nearest")
    data = img.get_fdata()
    return (data != 0) & ~np.isnan(data), img

def dice(a, b):
    inter = np.count_nonzero(a & b)
    return 2*inter / (np.count_nonzero(a) + np.count_nonzero(b))

def jaccard(a, b):
    inter = np.count_nonzero(a & b)
    union = np.count_nonzero(a | b)
    return inter / union if union > 0 else np.nan

def overlap_coeff(a, b):
    inter = np.count_nonzero(a & b)
    return inter / min(np.count_nonzero(a), np.count_nonzero(b))

# --- Load and binarize ---
bin_e, img_e = load_bin(paths["emotion"])
bin_s, _     = load_bin(paths["scene"], img_e)
bin_g, _     = load_bin(paths["gist"], img_e)

# --- Pairwise ---
pairs = [("emotion", bin_e, "scene", bin_s),
         ("emotion", bin_e, "gist", bin_g),
         ("scene",   bin_s, "gist", bin_g)]

print("=== Pairwise overlap ===")
for a_name, a, b_name, b in pairs:
    d = dice(a,b)
    j = jaccard(a,b)
    o = overlap_coeff(a,b)
    print(f"{a_name} vs {b_name}:")
    print(f"  Dice    = {d:.3f}")
    print(f"  Jaccard = {j:.3f}")
    print(f"  Overlap = {o:.3f}")
    print("")

# --- Three-way ---
inter3 = np.count_nonzero(bin_e & bin_s & bin_g)
union3 = np.count_nonzero(bin_e | bin_s | bin_g)
triple_iou = inter3/union3 if union3>0 else np.nan

print("=== Triple overlap ===")
print(f"Voxels in all three: {inter3}")
print(f"Voxels in union:     {union3}")
print(f"Triple IoU:          {triple_iou:.3f}")


=== Pairwise overlap ===
emotion vs scene:
  Dice    = 0.316
  Jaccard = 0.188
  Overlap = 0.466

emotion vs gist:
  Dice    = 0.079
  Jaccard = 0.041
  Overlap = 0.199

scene vs gist:
  Dice    = 0.065
  Jaccard = 0.034
  Overlap = 0.100

=== Triple overlap ===
Voxels in all three: 49
Voxels in union:     6406
Triple IoU:          0.008


C:\Users\yujunchen\AppData\Local\Temp\ipykernel_22096\4040873944.py:19: FutureWarning: 'force_resample' will be set to 'True' by default in Nilearn 0.13.0.
Use 'force_resample=True' to suppress this warning.
  img = resample_to_img(img, ref_img, interpolation="nearest")
C:\Users\yujunchen\AppData\Local\Temp\ipykernel_22096\4040873944.py:19: FutureWarning: From release 0.13.0 onwards, this function will, by default, copy the header of the input image to the output. Currently, the header is reset to the default Nifti1Header. To suppress this warning and use the new behavior, set `copy_header=True`.
  img = resample_to_img(img, ref_img, interpolation="nearest")
